In [ ]:
# read the csv file
import pandas as pd

# read the CSV file
df = pd.read_csv('data_optim_add 1.csv')  #

# Filter conditions
filtered_df = df[(df['year'] == 2023) & 
                 (df['compensatedRelativePressure_mafiltered'].notna()) &
                (df['alpha'] != 0)] 

# Step 3: Create a new DataFrame (grouped by csemCur, ensuring non-empty parts for each csemCur)
new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))

# load heritage data
donnees_patrimoniales = pd.read_csv('Donnees patrimoniales - volume.csv')

# convert csemCur to numeric for both DataFrames
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce')
donnees_patrimoniales['csemCur'] = pd.to_numeric(donnees_patrimoniales['csemCur'], errors='coerce')

# merge volume column based on csemCur (left join to keep all rows in new_df)
new_df = new_df.merge(
    donnees_patrimoniales[['csemCur', 'Volume']], 
    on='csemCur', 
    how='left'
)

# convert csemCur back to Int
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce').astype('Int64')  # use Int64 to allow for NaN values

# save to a new CSV
new_df.to_csv('new_df_with_volume.csv', index=False)

# results
print("New DataFrame (new_df) created successfully!")
print(f"Shape: {new_df.shape}")
print(f"Unique csemCur count: {new_df['csemCur'].nunique()}")
print(new_df.head())



New DataFrame (new_df) created successfully!
Shape: (595, 13)
Unique csemCur count: 595
      csemCur  year  delta_pressure   cr  \
0  5000200795  2023             NaN  0.0   
1  5000252448  2023             NaN  0.0   
2  5000252449  2023             NaN  0.0   
3  5000252450  2023             NaN  0.0   
4  5000252524  2023             NaN  0.0   

   compensatedRelativePressure_mafiltered  COPER  MANX     alpha  \
0                                3.628186      0     0  0.004915   
1                                3.208579      0     0  0.181613   
2                                3.483369      0     0  0.022172   
3                                3.452095      0     0  0.018647   
4                                3.502114      0     0  0.032852   

   algo_critere_exploit_B_mafiltered  algo_critere_env_B_mafiltered  \
0                              False                          False   
1                              False                          False   
2                        

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_46436/748567064.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))


In [74]:
# import pandas as pd

# df = pd.read_excel("Donnees patrimoniales - edited.xlsm")
# df = df.rename(columns={'N° eqt': 'csemCur'})
# df.to_csv('Donnees patrimoniales - volume.csv', index=False)

# print("✅ Donnees patrimoniales - volume.csv")
# print("columns:", df.columns.tolist())


In [ ]:
# Model 1: no constraint
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# read the CSV file
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation


T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2       # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

mu_COPER = 1.6
mu_MANT  = 4.2

# M = 0.2  
M = max(alpha_init)                        

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# # 3. alpha evolution 
# for i in range(N):
#     for t in range(1, T_years):  # t=1 to T_years-1, because we reference t-1
#         # 1. No maintenance: alpha[t] = 1.01 * alpha[t-1]
#         growth = 1.01 * alpha[i,t-1]
#         mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t] + x_MANT[i,t]))
#         mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t] + x_MANT[i,t]))
        
#         # 2. COPER maintenance: alpha[t] = alpha[t-1] / mu
#         maint_coper = alpha[i,t-1] / mu_COPER
#         mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t]))
#         mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t]))
        
#         # 3. MANT maintenance
#         maint_mant = alpha[i,t-1] / mu_MANT
#         mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t]))
#         mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t]))

# 3. alpha evolution 
for i in range(N):
    for t in range(1, T_years):  # t=1 to T_years-1, because we reference t-1
        # 1. No maintenance: alpha[t] = 1.01 * alpha[t-1]
        growth = 1.01 * alpha[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t-1] + x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t-1] + x_MANT[i,t-1]))
        
        # 2. COPER maintenance: alpha[t] = alpha[t-1] / mu
        maint_coper = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t]))
        
        # 3. MANT maintenance
        maint_mant = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process


# -------------- Solve & Output --------------
mdl.Params.MIPGap = 0.05   # 5% gap
mdl.optimize()

# analyze infeasibility
# print(f"Status code: {mdl.status}")

# print(f"GRB常量: {GRB.INFEASIBLE}, {GRB.OPTIMAL}")

# if mdl.status == GRB.INFEASIBLE:
#     mdl.computeIIS()
#     print("\n=== Conflict Constraints ===")
#     for c in mdl.getConstrs():
#         if c.IISConstr:
#             print(f"  {c.ConstrName}")

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Set parameter MIPGap to value 0.05
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 132 rows, 66 columns and 405 nonzeros
Model fingerprint: 0xacdb6cfe
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 4.139217e+08
Presolve removed 96 rows and 48 columns
Presolve time: 0.02s
Presolved: 36 rows, 18 columns, 120 nonzeros
Found heuristic solution: objective 4.08558

In [ ]:
# Model 2: budget constraint
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# read the CSV file
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

lpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  
C_leak_t = [c * 25200 for c in c_factors]  

Total_budget = 56700       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2

# M = 0.2                        # Big-M 
M = max(alpha_init) 

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution 
for i in range(N):
    for t in range(1, T_years):  # t=1 to T_years-1, because we reference t-1
        # 1. No maintenance: alpha[t] = 1.01 * alpha[t-1]
        growth = 1.01 * alpha[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # 2. COPER maintenance: alpha[t] = alpha[t-1] / mu
        maint_coper = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t]))
        
        # 3. MANT maintenance
        maint_mant = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# Budget constraint
mdl.addConstr(action_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.Params.MIPGap = 0.05   # 5% gap
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185, 0.018646529811208, 0.0328520079677214, 0.1135906463577228, 0.1442844871064215, 0.1179977537955345, 0.2886337307427016, 0.0807847481164319, 0.0173929192935642, 0.0151630310908607, 0.1051734316562682, 0.0038573082497245, 0.0069568026592355, 0.0046884483802392, 0.0134447131234082, 0.0186238682247815, 0.1527937655320737, 0.0347517193279847, 0.1351360945694349, 0.0233940176341845, 0.0069836995986677, 0.0083775190692959, 0.0117649991319785, 0.0328168758447618, 0.2095426215528058, 0.1219035255261631, 0.0533209308962354, 0.0112470096868016, 0.0004152576551435, 0.002762868853545, 0.0301377378481807, 0.0081591097913488, 0.0153777408031595, 0.0080165753239968, 0.0292268659530364, 0.0133666342497473, 0.0027120551724136, 0.0001785712582724, 0.0306591210386554, 0.165326644820168, 0.0466031667638488, 0.1609135127030445, 0.0953195748432587, 0.1039451266528637, 0.0022956841539589

In [87]:
# Model 3: resources constraint

from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# read the CSV file
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 3                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]


mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 2
# M = 0.2                        # Big-M (alpha range approximately 0~0.01)
M = max(alpha_init) 

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution 
for i in range(N):
    for t in range(1, T_years):  # t=1 to T_years-1, because we reference t-1
        # 1. No maintenance: alpha[t] = 1.01 * alpha[t-1]
        growth = 1.01 * alpha[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # 2. COPER maintenance: alpha[t] = alpha[t-1] / mu
        maint_coper = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t]))
        
        # 3. MANT maintenance
        maint_mant = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t]))

# 4. Total tasks limit
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.Params.MIPGap = 0.05   # 5% gap
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
Volume: [7.58361265949119, 63.1967721624266, 63.0536851311155]
Set parameter MIPGap to value 0.05
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 133 rows, 66 columns and 447 nonzeros
Model fingerprint: 0xb66e2c9b
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 2e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Found heuristic solution: objective 9.287137e+08
Presolve removed 24 rows and 12 columns
Presolve time: 0.00s
Presolved: 109 rows, 54 columns, 392 nonzeros
Variable types: 18 continuous, 36 integer 

In [89]:
# Model 4: budget and resources constraints
from gurobipy import Model, GRB, quicksum

# -------------- Data Preparation --------------
# read the CSV file
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

# ---------------- Parameters ----------------
N = 595                          # CSEM number

alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
print("Volume:", V) 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]    # K[i] calculation

T_years = 7                   # planning years
Y = 365                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 11245500       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2

L_total = 302
# M = 0.2                        # Big-M
M = max(alpha_init) 

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution 
for i in range(N):
    for t in range(1, T_years):  # t=1 to T_years-1, because we reference t-1
        # 1. No maintenance: alpha[t] = 1.01 * alpha[t-1]
        growth = 1.01 * alpha[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # 2. COPER maintenance: alpha[t] = alpha[t-1] / mu
        maint_coper = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t]))
        
        # 3. MANT maintenance
        maint_mant = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t]))

# Total tasks limit
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)


# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# Budget constraint
mdl.addConstr(action_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.Params.MIPGap = 0.05   # 5% gap
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

csemCur number: [5000200795, 5000252448, 5000252449, 5000252450, 5000252524, 5000252525, 5000252528, 5000252529, 5000252530, 5000252531, 5000252532, 5000252544, 5000252545, 5000252549, 5000252551, 5000252554, 5000252555, 5000261916, 5000270722, 5000270810, 5000279236, 5000279237, 5000279238, 5000279239, 5000279241, 5000279256, 5000279257, 5000279258, 5000279260, 5000279263, 5000764620, 5000764621, 5000764622, 5000764623, 5000764624, 5000764625, 5000764632, 5000764633, 5000764634, 5000808065, 5000808066, 5000808067, 5000808086, 5000808087, 5000808088, 5000818455, 5000818456, 5000818458, 5000818459, 5000818460, 5000818461, 5000818462, 5000818463, 5000818464, 5000818487, 5000818489, 5000818490, 5000818491, 5000818492, 5000818493, 5000818508, 5000818509, 5000818510, 5000818612, 5000818614, 5000818616, 5000818622, 5000818628, 5000818629, 5000818630, 5000818631, 5000818632, 5000818634, 5000818644, 5000818645, 5000818646, 5000818647, 5000818648, 5000818656, 5000818657, 5000818658, 5000818659,